# 📦 Notebook 01 — BigQuery Bronze Layer Ingestion

**Project:** Plume Care Navigator

This notebook loads the four synthetic Parquet files generated in `00_data_generation.ipynb` into Google BigQuery as the **Bronze (raw ingest) layer**. It also validates row counts, schema, and referential integrity after loading.

### Prerequisites
- Notebook `00_data_generation.ipynb` has been run and the four Parquet files are in `/content/plume_data/`
- A Google Cloud project with BigQuery API enabled
- A BigQuery dataset named `plume_bronze` already created

### What this notebook does
1. Authenticates with Google Cloud
2. Loads all four Parquet files into BigQuery Bronze tables
3. Runs post-load validation queries
4. Prints a summary of the loaded data

## 1. Configuration

In [ ]:
# ── Update these values for your GCP project ──────────────────────────────
PROJECT_ID = 'YOUR_PROJECT_ID'   # e.g. 'plume-care-navigator-demo'
DATASET    = 'plume_bronze'      # Must already exist in BigQuery
DATA_DIR   = '/content/plume_data'

TABLES = {
    'patients':         'patients.parquet',
    'subscriptions':    'subscriptions.parquet',
    'lab_results':      'lab_results.parquet',
    'clinical_visits':  'clinical_visits.parquet',
}

print(f'Project  : {PROJECT_ID}')
print(f'Dataset  : {DATASET}')
print(f'Data dir : {DATA_DIR}')

## 2. Authentication & Client Setup

In [ ]:
from google.colab import auth
from google.cloud import bigquery
import os
import pyarrow.parquet as pq

auth.authenticate_user()
client = bigquery.Client(project=PROJECT_ID)
print(f'✓ Authenticated as project: {PROJECT_ID}')

## 3. Load Parquet Files into BigQuery Bronze

In [ ]:
job_config = bigquery.LoadJobConfig(
    source_format=bigquery.SourceFormat.PARQUET,
    write_disposition=bigquery.WriteDisposition.WRITE_TRUNCATE,  # Idempotent: safe to re-run
    autodetect=True,
)

load_results = {}

for table_name, filename in TABLES.items():
    table_id  = f'{PROJECT_ID}.{DATASET}.{table_name}'
    file_path = os.path.join(DATA_DIR, filename)

    if not os.path.exists(file_path):
        print(f'  ✗ File not found: {file_path} — run notebook 00 first.')
        continue

    local_rows = pq.read_metadata(file_path).num_rows
    print(f'Loading {filename} ({local_rows:,} rows) → {table_id} ...')

    with open(file_path, 'rb') as f:
        job = client.load_table_from_file(f, table_id, job_config=job_config)

    job.result()  # Block until complete

    if job.errors:
        print(f'  ✗ Errors: {job.errors}')
    else:
        bq_table = client.get_table(table_id)
        load_results[table_name] = {'local': local_rows, 'bq': bq_table.num_rows}
        match = '✓' if local_rows == bq_table.num_rows else '⚠ MISMATCH'
        print(f'  {match} Loaded {bq_table.num_rows:,} rows into {table_id}')

print('\nAll tables loaded.')

## 4. Post-Load Validation

In [ ]:
import pandas as pd

print('── Row Count Validation ──')
for table_name in TABLES:
    query = f'SELECT COUNT(*) AS row_count FROM `{PROJECT_ID}.{DATASET}.{table_name}`'
    result = client.query(query).result()
    count = list(result)[0].row_count
    print(f'  {table_name:<25} {count:>12,} rows')

In [ ]:
print('── Null Check: patients table ──')
null_query = f"""
SELECT
  COUNTIF(patient_id      IS NULL) AS null_patient_id,
  COUNTIF(state           IS NULL) AS null_state,
  COUNTIF(gender_identity IS NULL) AS null_gender,
  COUNTIF(hrt_regimen     IS NULL) AS null_hrt,
  COUNTIF(birth_date      IS NULL) AS null_birth_date,
  COUNTIF(start_date      IS NULL) AS null_start_date
FROM `{PROJECT_ID}.{DATASET}.patients`
"""
df = client.query(null_query).to_dataframe()
print(df.T.rename(columns={0: 'null_count'}).to_string())

In [ ]:
print('── Referential Integrity: orphan patient_ids ──')
for child_table in ['subscriptions', 'lab_results', 'clinical_visits']:
    orphan_query = f"""
    SELECT COUNT(*) AS orphan_count
    FROM `{PROJECT_ID}.{DATASET}.{child_table}` c
    WHERE NOT EXISTS (
        SELECT 1 FROM `{PROJECT_ID}.{DATASET}.patients` p
        WHERE p.patient_id = c.patient_id
    )
    """
    result = client.query(orphan_query).result()
    orphans = list(result)[0].orphan_count
    status = '✓' if orphans == 0 else f'⚠ {orphans:,} orphans found'
    print(f'  {child_table:<25} {status}')

In [ ]:
print('── Business Logic: Subscription MRR ──')
mrr_query = f"""
SELECT
  status,
  plan_type,
  COUNT(*)                   AS patient_count,
  ROUND(SUM(mrr), 2)         AS total_mrr,
  ROUND(AVG(mrr), 2)         AS avg_mrr
FROM `{PROJECT_ID}.{DATASET}.subscriptions`
GROUP BY status, plan_type
ORDER BY status, plan_type
"""
df = client.query(mrr_query).to_dataframe()
print(df.to_string(index=False))

active_mrr = df[df['status'] == 'Active']['total_mrr'].sum()
print(f'\nTotal Active MRR  : ${active_mrr:>12,.2f} / month')
print(f'Annualised ARR    : ${active_mrr * 12:>12,.2f} / year')

In [ ]:
print('── Lab Results: Flag Distribution ──')
flag_query = f"""
SELECT
  test_name,
  flag,
  COUNT(*) AS result_count,
  ROUND(AVG(result_value), 2) AS avg_value
FROM `{PROJECT_ID}.{DATASET}.lab_results`
GROUP BY test_name, flag
ORDER BY test_name, flag
"""
df = client.query(flag_query).to_dataframe()
print(df.to_string(index=False))

## 5. Next Steps

The Bronze layer is now loaded and validated. The next step is to run the **dbt models** to transform the data through the Silver and Gold layers.

From your local machine (or a Colab cell with `%%bash`):
```bash
cd dbt/
dbt deps
dbt build --select bronze silver gold mart
dbt docs generate && dbt docs serve
```

Then proceed to `02_rag_pipeline.ipynb` to build the ChromaDB vector store.